<!-- # <span style="color:#FFFFFF; font-size: 0%;">1 | Introduction</span>
<div style="border-radius: 0px; background-color: #112241; text-align:center;">
    <h4 style="color: white; padding: 1.5rem; font-size: 19px"><b>1 | INTRODUCTION</b></h4>
</div>
<!-- <br> -->

<div style="display: flex; flex-direction: row; align-items: center;">
    <div style="flex: 0; margin-top: 8px;">
        <img src="https://irp.nih.gov/sites/default/files/styles/image_half/public/media/image/2022-01/Rudd%20Center%203%20cropped.jpg?itok=eAs3MJNe" alt="Image" style="max-width: 300px; max-height: 300px;" />
    </div>
    <div style="flex: 1; margin-left: 30px; margin-top: 6px">
        <p style="font-weight: bold; color: black; font-size: 17px">Introduction</p>
        <p>This notebook is created for Multiclass Classification with a Obesity Risk data in the Playground Season-4 Episode-2.
        </p>
        <p>This is a beginner-friendly notebook that attempts to perform Exploratory Data Analysis on the Obesity Risk Dataset and eventually train a <b>LightGBM</b> model on it and enhance the predictions by fine-tuning the model.
        </p>
        <p>Let's explore and then make results and discussion to gain deeper insights from our analysis. Let's explore and then make results and discussion to gain deeper insights from our analysis.</p>
        <blockquote>  If find this notebook helpful please consider upvoting ❤️</blockquote>
    </div>
</div>


## Contents:
<hr>

1. [Data Exploration](#data)
2. [Exploritory Data Analysis](#eda)
3. [Modeling](#model)
4. [Hyperparameter Tuning and Cross Validation](#hyper)
5. [Visualizations](#graph)
6. [Submission](#submission)

### All the used libraries:

- Numpy
- Pandas
- Matplotlib
- Seaborn
- Scikit-learn
- LighGBM
- Optuna
- warnings

### Models used to make predictions:

- LighGBM Classifier
- Optuna for Hyperparameter tuning

Now, let's import the data.

<hr>

### Update: Tried some new features like and tried this features with new set of parameters.
- BMI [Body Mass Index] = Weight / Height**2 
- WIR [Water Intake Ratio] = Weight / CH2O 
- STR [Sedentary Time Ratio] = FAF / TUE

In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report

from optuna.samplers import TPESampler
import optuna

import warnings
warnings.filterwarnings("ignore")

In [ ]:
rc = {
    "axes.facecolor": "#F8F8F8",
    "figure.facecolor": "#F8F8F8",
    "axes.edgecolor": "#000000",
    "grid.color": "#EBEBE7" + "30",
    "font.family": "serif",
    "axes.labelcolor": "#000000",
    "xtick.color": "#000000",
    "ytick.color": "#000000",
    "grid.alpha": 0.4,
}

sns.set(rc=rc)
palette = ['#302c36', '#037d97', '#E4591E', '#C09741',
           '#EC5B6D', '#90A6B1', '#6ca957', '#D8E3E2']

from colorama import Style, Fore
blk = Style.BRIGHT + Fore.BLACK
mgt = Style.BRIGHT + Fore.MAGENTA
red = Style.BRIGHT + Fore.RED
blu = Style.BRIGHT + Fore.BLUE
res = Style.RESET_ALL

plt.style.use('seaborn-v0_8-pastel')


# 1. Data Exploration: <a name="data"></a>
The dataset for this competition (both train and test) was generated from a deep learning model trained on the [Obesity or CVD risk dataset](https://www.kaggle.com/datasets/aravindpcoder/obesity-or-cvd-risk-classifyregressorcluster/data).

> Submissions are evaluated using the accuracy score so you must evaluate your models on the basis of accuracy score.

## 1.1 Data Description:

Here is a quick description of the data which was taken from the original data.

### Features:
- FAVC: Frequent consumption of high caloric food,
- FCVC: Frequency of consumption of vegetables,
- NCP: Number of main meals,
- CAEC: Consumption of food between meals,
- CH20: Consumption of water daily,
- CALC: Consumption of alcohol.
- SCC: Calories consumption monitoring,
- FAF: Physical activity frequency,
- TUE: Time using technology devices,
- MTRANS: Transportation used.
- Gender,
- Age,
- Height and
- Weight.

> In Original data 'TUE', 'FAF', 'CH2O', 'NCP', 'FCVC' columns are categorical with 2 or 3 classes but in the competition data they are numerical so you can keep them as it is or round them to nearest integers to turn them into categorical varibles.

### Targets:
- Underweight: Less than 18.5
- Normal: 18.5 to 24.9
- Overweight: 25.0 to 29.9
- Obesity I: 30.0 to 34.9
- Obesity II: 35.0 to 39.9
- Obesity III: Higher than 40


In [ ]:
train_data = pd.read_csv('/kaggle/input/playground-series-s4e2/train.csv')
test_data = pd.read_csv('/kaggle/input/playground-series-s4e2/test.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s4e2/sample_submission.csv')

original_data = pd.read_csv('/kaggle/input/obesity-or-cvd-risk-classifyregressorcluster/ObesityDataSet.csv')

## 1.2 Train Data

In [ ]:
train_data.head(10)

In [ ]:
train_data.describe().T.style.background_gradient()

## 1.2 Test Data

In [ ]:
test_data.head()

In [ ]:
test_data.describe().T.style.background_gradient()

## 1.2 Original Data

In [ ]:
original_data.head()

In [ ]:
original_data.describe().T.style.background_gradient()

# 2. Exploritory Data Analysis <a name="eda"></a>
- Exploratory Data Analysis (EDA) is an analysis approach that identifies general patterns in the data. These patterns include outliers and features of the data that might be unexpected. EDA is an important first step in any data analysis.

## 2.1 Null Values:

Missing data/Null values is defined as the values or data that is not stored (or not present) for some variable/s in the given dataset.Here is a list of popular strategies to handle missing values in a dataset
- Deleting the Missing Values
- Imputing the Missing Values
- Imputing the Missing Values for Categorical Features
- Imputing the Missing Values using Sci-kit Learn Library
- Using “Missingness” as a Feature

Let's see if our data has any missing values or not.

In [ ]:
sns.displot(data=train_data.isnull().melt(value_name='missing'),
    y='variable',
    hue='missing',
    multiple='fill',
    height=8,
#     width=10,
    aspect=1.6
)

# specifying a threshold value
plt.axvline(0.4, color='r')
plt.title('Null Values in Train Data', fontsize=13)
plt.show()

# -------------------------

sns.displot(data=test_data.isnull().melt(value_name='missing'),
    y='variable',
    hue='missing',
    multiple='fill',
    height=8,
#     width=10,
    aspect=1.6
)

# specifying a threshold value
plt.axvline(0.4, color='r')
plt.title('Null Values in Test Data', fontsize=13)
plt.show()

As we can see we have no null values in the both `train` and `test` data.

## 2.2 Target Variable Analysis

In [ ]:
f,ax=plt.subplots(1,2,figsize=(19,8))
train_data['NObeyesdad'].value_counts().plot.pie(autopct='%1.1f%%',ax=ax[0],shadow=True)
# ax[0].set_title('Pie-Plot')
ax[0].set_ylabel('')
sns.countplot(x='NObeyesdad',data=train_data,ax=ax[1])
plt.xticks(rotation=90)
# ax[1].set_title('Count-Plot')
plt.suptitle('Target Value Anaysis - Competition Data')
plt.show()

In [ ]:
f,ax=plt.subplots(1,2,figsize=(19,8))
original_data['NObeyesdad'].value_counts().plot.pie(autopct='%1.1f%%',ax=ax[0],shadow=True)
# ax[0].set_title('Pie-Plot')
ax[0].set_ylabel('')
sns.countplot(x='NObeyesdad',data=original_data,ax=ax[1])
plt.xticks(rotation=90)
# ax[1].set_title('Count-Plot')
plt.suptitle('Target Value Anaysis - Original Data')
plt.show()

In [ ]:
# Unique value counts for each column
unique_counts = train_data.nunique()

# Threshold to distinguish continuous and categorical
threshold = 10

continuous_vars = unique_counts[unique_counts > threshold].index.tolist()
categorical_vars = unique_counts[unique_counts <= threshold].index.tolist()

# Removing the 'outcome' from categorical since it's our target variable
if 'outcome' in categorical_vars:
    categorical_vars.remove('outcome')
if 'id' in continuous_vars:
    continuous_vars.remove('id')

print(f"Categorical Variables: {categorical_vars}")
print(f"Continousl/Numerical Variables: {continuous_vars}")

## 2.3 Categorical Variables Analysis:    
In statistics, a categorical variable (also called qualitative variable) is a variable that can take on one of a limited, and usually fixed, number of possible values, assigning each individual or other unit of observation to a particular group or nominal category on the basis of some qualitative property. Categorical data is the statistical data type consisting of categorical variables or of data that has been converted into that form.

In our data categorical varibles are:

- Gender
- family_history_with_overweight 
- FAVC 
- CAEC 
- SMOKE 
- SCC 
- CALC 
- MTRANS
- NObeyesdad

In [ ]:
categorical_vars.remove('NObeyesdad')

for column in categorical_vars:
    f,ax=plt.subplots(1,2,figsize=(18,5.5))
    train_data[column].value_counts().plot.pie(autopct='%1.1f%%',ax=ax[0],shadow=True)
    ax[0].set_ylabel(f'{column}')
    sns.countplot(x=column,data=train_data,ax=ax[1])
    plt.xticks(rotation=90)
    plt.suptitle(f'{column}')
    plt.show()

### Some Observations from above plots:

- Most of the variables like `CAEC`, `SMOKE` and `transportation_mode` are highly imbalanced.
- In Origional data only `Age`, `Height` and `Width` were continuous data but in here there are many features that are continuous and not categorical.


## 2.4 Numerical Value Analysis:
In Mathematics, if a variable can take on two or more distinct real values so that it can also take all real values between them (even values that are randomly close together). In this case, the variable is continuous in the given interval. Continuous data is the statistical data type consisting of continuous variables or of data that has been converted into that form.

In our data Continuous variables are:

- Age 
- Height 
- Weight 
- FCVC 
- NCP 
- CH2O 
- FAF 
- TUE

In [ ]:
for column in continuous_vars:
    fig, ax = plt.subplots(figsize=(18, 4))
    fig = sns.histplot(data=train_data, x=column, hue="NObeyesdad", bins=50, kde=True)
    plt.ylim(0,500)
    plt.show()

### Some Observations from above plots:

- As I discussed earlier some of the parameters were categorical in original data and that's why some of the features like `TUE`, `FAF`, `CH20` are skewed at the integers.
- Other variables `Age`, `Height` and `Weight` does not show any kind of skewness.
- Distribution of data for is also almost same for all the variables.
- As of now I have ketp the categorical and continuous varibles as it is.

## 2.5 Multivariate Analysis:
Multivariate analysis is based in observation and analysis of more than one statistical outcome variable at a time.

In [ ]:
df3 = train_data[['Age', 'Weight', 'Height', 'NObeyesdad']].copy()

sns.pairplot(df3, hue="NObeyesdad", corner=True, size=4)
plt.show()

## 2.6 Correlation Analysis:

Correlation is the statistical analysis of the relationship or dependency between two variables. Correlation allows us to study both the strength and direction of the relationship between two sets of variables.

There are mainly 3 types of Correlations:

- Positive Correlation: Two variables are said to be positively correlated when their values move in the same direction.
- Neutral Correlation: No relationship in the change of variables X and Y. In this case, the values are completely random and do not show any sign of correlation.
- Negative Correlation: Finally, variables X and Y will be negatively correlated when their values change in opposite directions.


In [ ]:
df = train_data[continuous_vars].copy()

corr_matrix=df.corr()

mask = np.zeros_like(corr_matrix)
mask[np.triu_indices_from(mask)] = True

f,ax=plt.subplots(figsize=(15,11))
sns.heatmap(corr_matrix, mask=mask, annot=True)
plt.suptitle('Correlation Matrix')
plt.show()

# 3. Modelling <a name="model"></a>

I will be using LightGBM model for this data.

<div style="display: flex; flex-direction: row; align-items: center;">
    <div style="flex: 0; margin-top: 8px;">
        <img src="https://pbs.twimg.com/media/DU5uczNW0AE2vNF.jpg" alt="Image" style="max-width: 250px;" />
    </div>
    <div style="flex: 1; margin-left: 30px; margin-top: 6px">
        <p style="font-weight: bold; color: black; font-size: 17px">LightGBM:</p>
        <p><b>LightGBM</b>, short for light gradient-boosting machine, is a free and open-source distributed gradient-boosting framework for machine learning, originally developed by Microsoft. </p>
        <p>LightGBM supports both classification and regression tasks, and is known for its high speed and accuracy. It is often used in machine learning competitions, and is a popular choice for Kaggle users.</p>
        <p>LightGBM has lots of advantages over other gradient boosting frameworks. It’s fast, scalable, and has a lower memory usage than XGBoost. It also has a higher accuracy than other frameworks, and is able to handle large datasets.</p>
    </div>
</div>

- LightGBM uses a technique called gradient boosting, which combines multiple weak learners (usually decision trees) to create a strong predictive model.
- LightGBM splits the tree leaf-wise as opposed to other boosting algorithms that grow tree level-wise. It chooses the leaf with the maximum delta loss to grow. Since the leaf is fixed, the leaf-wise algorithm has a lower loss compared to the level-wise algorithm. Leaf-wise tree growth might increase the complexity of the model and may lead to overfitting in small datasets.

<img src="https://miro.medium.com/v2/resize:fit:786/format:webp/1*AZsSoXb8lc5N6mnhqX5JCg.png" alt="Image" style="max-width: 2000px; max-height: 500px;" />
Other Gradient Boosting Algorithms

<img src="https://miro.medium.com/v2/resize:fit:786/format:webp/1*whSa8rY4sgFQj1rEcWr8Ag.png" alt="Image" style="max-width: 2000px; max-height: 500px;" />
LigbGBM Algorithm

In [ ]:
X = train_data.drop(['id', 'NObeyesdad'], axis=1)
y = train_data['NObeyesdad']

In [ ]:
X.head()

In [ ]:
y.head()

## 3.3 Creating some features:
As everyone already know BMI is a very popular and effective feature for this competition so along with that I will also try creating some other features details of that all are given below:

- BMI [Body Mass Index] = Weight / Height**2
- WIR [Water Intake Ratio] = Weight / CH2O
- STR [Sedentary Time Ratio] = FAF / TUE
- MR [Meal Regularity] = NCP + CAEC [!] Transform NCP to categorical feature before
- PAL [Physical Activity Level] = FCVC * FCVC_weight + FAF * FAF_weight. Normalize values to be in range [0, 1] before computing.

Reference: https://www.kaggle.com/competitions/playground-series-s4e2/discussion/473273

### 1. BMI [Body Mass Index]:

In [ ]:
X['BMI'] = X['Weight'] / (X['Height'] ** 2)
test_data['BMI'] = test_data['Weight'] / (test_data['Height'] ** 2)
X.head()

### 2. WIR [Water Intake Ratio]

In [ ]:
X['WIR'] = X['Weight'] / X['CH2O']                                # Weight / CH2O
test_data['WIR'] = test_data['Weight'] / test_data['CH2O']
X.head()

### 3. STR [Sedentary Time Ratio]

In [ ]:
X['STR'] = X['FAF'] / X['TUE']                        #  FAF / TUE
test_data['STR'] = test_data['FAF'] / test_data['TUE']
X.head()

## 3.3 Encoding Caegorical Variables:

There are multiple encoders available but 2 of them are very famous.

**1. Label Encoder:**
- Label Encoding is a popular encoding technique for handling categorical variables. A unique integer or alphabetical ordering represents each label.
- Problems with Label Encoder: Although if our Categorical Data has no order in it the `LabelEncoder` will assign the integer according to the alphabetical ordering and because of that.

**2. One Hot Encoder:**
- One-Hot Encoding is another popular technique for treating categorical variables. It simply creates additional features based on the number of unique values in the categorical feature. Every unique value in the category will be added as a feature. One-Hot Encoding is the process of creating dummy variables.

- In the case of LGBM OneHotEncoder works better than the other encoders so I will use that.

In [ ]:
X_encoded = pd.get_dummies(X, columns=['MTRANS',
                                       'SCC',
                                       'SMOKE',
                                       'CAEC',
                                       'FAVC',
                                       'family_history_with_overweight',
                                       'Gender'])

X_encoded.head()

In [ ]:
test_data = test_data.drop(['id'], axis=1)

X_test_encoded = pd.get_dummies(test_data, columns=['MTRANS',
                                       'SCC',
                                       'SMOKE',
                                       'CAEC',
                                       'FAVC',
                                       'family_history_with_overweight',
                                       'Gender'])

X_test_encoded.head()

> Here CALC has 3 values in train data `sometimes`, `frequently` and `no` but in test data it has 4 values `sometimes`, `frequently`, `no` and `always` by applying one hot encoding on that will increase the feature by 1 so I have done the LabelEncoding on the `CALC` feature.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

X_encoded.CALC = le.fit_transform(X_encoded.CALC)
X_test_encoded.CALC = le.fit_transform(X_test_encoded.CALC)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, 
                                                    y, 
                                                    random_state=42, 
                                                    stratify=y)

As we can see we have both `train` and `test` data in equal proportion.

## 3.5 Creating Baseline Model:

- Now that we have splitted the data into `train` and `test` set sucessfully. Let's create a baseline model for our data. We will try to improve the model by **Hyperparameter Tuning** and **Cross-Validation**.
- I have not defined any parameters in the baseline model.

In [ ]:
base_model = lgb.LGBMClassifier()
base_model.fit(X_train, y_train)

y_pred = base_model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

# 4. Hyperparameter Tuning with Optuna: <a name="hyper"></a>

## 4.1 Parameters:
Basic understanding of all the parameters used here is given below:

**1. Objective:**

The three most popular objectives are:

- **`binary`** - binary classification (the target contains only two classes, i.e., cat or dog)
- **`multiclass`** - multi-class classification (more than two classes in the target, i.e., apple/orange/banana)
- **`Regression`** - for regression problems

Our data has 7 classes so I am using `multiclass` as objective here.

**2. Verobse:** To know what is going on in the model train we set `verbose` as -1. 

**3. Boosting Type:** `gbdt` for traditional Gradient Boosting Decision Tree

**4. num_classes:** Number of classes (here 7).

**5. lambda:** lambda specifies regularization. Typical value ranges from 0 to 1.

**6. feature_fraction:** Used when your boosting is random forest. 0.8 feature fraction means LightGBM will select 80% of parameters randomly in each iteration for building trees.

**7. bagging_fraction:** specifies the fraction of data to be used for each iteration and is generally used to speed up the training and avoid overfitting.

**8. num_leaves:** number of leaves in full tree, default: 31

**9. bagging_freq:** Frequency of randomly Bagging Sampling

`trial.suggest_float` selects between the min and max values provided in a continuous manner.

In [ ]:
def objective(trial):
    """
    Objective function to be minimized.
    """
    param = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "num_class": 7,
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }
    
    gbm = lgb.LGBMClassifier(**param)
    gbm.fit(X_train, y_train)
    preds = gbm.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    return accuracy

## 4.2 Optuna:

**1. TPESampler:** Sampler which uses TPE (Tree-structured Parzen Estimator) algorithm.

**2. Study name:** on which model you want to study/tune the parameters.

**3. Direction:** Minimize or Maximize the metric value.

**4. Trials:** Number of iterations.


In [ ]:
# sampler = TPESampler(seed=1)
# study = optuna.create_study(study_name="lightgbm", direction="maximize", sampler=sampler)
# study.optimize(objective, n_trials=200)

The Tuning process is very time-consuming and that's why I have commented that part.

In [ ]:
# print('Best parameters:', study.best_params)

> Best parameters: {'lambda_l1': 4.7516639792363974e-08, 'lambda_l2': 9.788847162667828, 'num_leaves': 46, 'feature_fraction': 0.6131121082599934, 'bagging_fraction': 0.9677622146367539, 'bagging_freq': 6, 'min_child_samples': 35}

In [ ]:
# print('Best value:', study.best_value)

> Best value: 0.9152215799614644

In [ ]:
# print('Best trial:', study.best_trial)

> Best trial: FrozenTrial(number=184, state=1, values=[0.9152215799614644], datetime_start=datetime.datetime(2024, 2, 1, 14, 9, 42, 916529), datetime_complete=datetime.datetime(2024, 2, 1, 14, 9, 45, 437998), params={'lambda_l1': 4.7516639792363974e-08, 'lambda_l2': 9.788847162667828, 'num_leaves': 46, 'feature_fraction': 0.6131121082599934, 'bagging_fraction': 0.9677622146367539, 'bagging_freq': 6, 'min_child_samples': 35}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lambda_l1': FloatDistribution(high=10.0, log=True, low=1e-08, step=None), 'lambda_l2': FloatDistribution(high=10.0, log=True, low=1e-08, step=None), 'num_leaves': IntDistribution(high=256, log=False, low=2, step=1), 'feature_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_fraction': FloatDistribution(high=1.0, log=False, low=0.4, step=None), 'bagging_freq': IntDistribution(high=7, log=False, low=1, step=1), 'min_child_samples': IntDistribution(high=100, log=False, low=5, step=1)}, trial_id=184, value=None)

## 4.3 Final Model:
- Now that we have the best parameter we will create the model out of that. 

In [ ]:
model = lgb.LGBMClassifier(lambda_l1=4.7516639792363974e-08, 
                           lambda_l2= 9.788847162667828, 
                           num_leaves= 46, 
                           feature_fraction= 0.6131121082599934, 
                           bagging_fraction= 0.9677622146367539, 
                           bagging_freq= 6, 
                           min_child_samples= 35)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

## 4.4 Trying New Model:
Instead of using the above parameters I want to try some new parameters and that's why I have taken another set of parameters from here: https://www.kaggle.com/code/moazeldsokyx/pgs4e2-highest-score-lgbm-hyperparameter-tuning/notebook

```
best_params = {
    "objective": "multiclass",          # Objective function for the model
    "metric": "multi_logloss",          # Evaluation metric
    "verbosity": -1,                    # Verbosity level (-1 for silent)
    "boosting_type": "gbdt",            # Gradient boosting type
    "random_state": 42,       # Random state for reproducibility
    "num_class": 7,                     # Number of classes in the dataset
    'learning_rate': 0.030962211546832760,  # Learning rate for gradient boosting
    'n_estimators': 500,                # Number of boosting iterations
    'lambda_l1': 0.009667446568254372,  # L1 regularization term
    'lambda_l2': 0.04018641437301800,   # L2 regularization term
    'max_depth': 10,                    # Maximum depth of the trees
    'colsample_bytree': 0.40977129346872643,  # Fraction of features to consider for each tree
    'subsample': 0.9535797422450176,    # Fraction of samples to consider for each boosting iteration
    'min_child_samples': 26             # Minimum number of data needed in a leaf
}
```

In [ ]:
params = {
    "objective": "multiclass",          # Objective function for the model
    "metric": "multi_logloss",          # Evaluation metric
    "verbosity": -1,                    # Verbosity level (-1 for silent)
    "boosting_type": "gbdt",            # Gradient boosting type
    "random_state": 42,       # Random state for reproducibility
    "num_class": 7,                     # Number of classes in the dataset
    'learning_rate': 0.030962211546832760,  # Learning rate for gradient boosting
    'n_estimators': 500,                # Number of boosting iterations
    'lambda_l1': 0.009667446568254372,  # L1 regularization term
    'lambda_l2': 0.04018641437301800,   # L2 regularization term
    'max_depth': 10,                    # Maximum depth of the trees
    'colsample_bytree': 0.40977129346872643,  # Fraction of features to consider for each tree
    'subsample': 0.9535797422450176,    # Fraction of samples to consider for each boosting iteration
    'min_child_samples': 26             # Minimum number of data needed in a leaf
}

In [ ]:
model_v2 = lgb.LGBMClassifier(**params)
model_v2.fit(X_train, y_train)

In [ ]:
y_pred_v2 = model_v2.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred_v2)

As we can see we are increasing our accuracy from 0.90% to 0.91% which is very good. I still haven't created all the features because it's decreasing the accuracy so I will again update the kernel if I make some progress.

# 5. Visualizations

## 5.1 LGBM Tree

In [ ]:
# fig, ax = plt.subplots(figsize=(20,10), sharex=True)
lgb.plot_tree(model_v2, tree_index=0,dpi=300, orientation='vertical')
plt.show()

## 5.2 Feature Importance

In [ ]:
feature_importance = model_v2.feature_importances_
sorted_idx = np.argsort(feature_importance)
fig = plt.figure(figsize=(18, 6))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(X_test.columns)[sorted_idx])
plt.title('Feature Importance')
plt.show()

# 6. Submission <a name="submission"></a>

In [ ]:
y_pred = model_v2.predict(X_test_encoded)

In [ ]:
sample_submission['NObeyesdad'] = y_pred
sample_submission.head()

In [ ]:
sample_submission.to_csv("submission.csv", index=False)

<div style="border-radius:10px;border:#112241 solid;padding: 15px;background-color:#ffffff00;font-size:100%;text-align:left">
    <b>Note:</b> Other models and parameters can also give the better result. This is not the best model but still we are getting pretty good results with it. If you find this notebook helpful please consider upvoting and check out my other works too❤️. If you have any suggestions let me know in the comments 🙂  </div>
    
- Author - Akhil
- Date - 2/1/2024